# Simulation Engine
Core simulation logic for the robot-based plant observation system.

In [ ]:
# Simulation constants
import threading
import time
import math

# Tile allocation based on paper
STORAGE_COLS = list(range(0, 8))  # Columns 0-7 for storage
EM_EXPOSURE_COLS = list(range(8, 12))  # Columns 8-11 for EM exposure

# Robot configuration
ROBOT_SPEED = 0.5  # columns per tick
OBSERVATION_TICKS = 3
PICKUP_TICKS = 2
PUTDOWN_TICKS = 2
OBSERVATION_COOLDOWN_TICKS = 30

# EM exposure constants
EM_EXPOSURE_TICKS = 100  # ticks for EM exposure
MAX_RETESTS = 5  # max times to retest same config

In [ ]:
def create_initial_simulation_state():
    """Create fresh simulation state"""
    return {
        'running': False,
        'tick': 0,
        'speed': 1.0,
        'robots': [
            {'id': 0, 'row': 0, 'col': 0.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0},
            {'id': 1, 'row': 1, 'col': 0.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0}
        ],
        'observation_queue': [],
        'observation_station': {'row': -1, 'col': 11},
        'plants_observed': [],
        'last_tick_time': None
    }

simulation_state = create_initial_simulation_state()
simulation_lock = threading.Lock()
simulation_thread = None

In [ ]:
def get_tile_allocation():
    """Return tile allocation info for frontend"""
    return {
        'storage_cols': STORAGE_COLS,
        'em_exposure_cols': EM_EXPOSURE_COLS,
        'observation_station': simulation_state['observation_station']
    }

def calculate_travel_time(from_col, to_col):
    """Calculate ticks needed to travel between columns"""
    distance = abs(to_col - from_col)
    return math.ceil(distance / ROBOT_SPEED)

def find_available_robot(row):
    """Find an idle robot for the given row"""
    for robot in simulation_state['robots']:
        if robot['row'] == row and robot['state'] == 'idle':
            return robot
    return None

In [ ]:
def update_robot(robot, simulation_state, get_sequencer_grid_fn, set_sequencer_grid_fn):
    """Update a single robot's state for one tick"""
    if robot['state'] == 'idle':
        return
    
    if robot['state'] == 'moving_to_pickup':
        if robot['target_col'] is not None:
            if robot['col'] < robot['target_col']:
                robot['col'] = min(robot['col'] + ROBOT_SPEED, robot['target_col'])
            elif robot['col'] > robot['target_col']:
                robot['col'] = max(robot['col'] - ROBOT_SPEED, robot['target_col'])
            
            if robot['col'] == robot['target_col']:
                robot['state'] = 'picking_up'
                robot['ticks_remaining'] = PICKUP_TICKS
    
    elif robot['state'] == 'picking_up':
        robot['ticks_remaining'] -= 1
        if robot['ticks_remaining'] <= 0:
            grid = get_sequencer_grid_fn()
            col = int(robot['col'])
            if grid[robot['row']][col]:
                robot['holding_plant'] = grid[robot['row']][col]
                grid[robot['row']][col] = ""
                set_sequencer_grid_fn(grid)
            
            # Move to observation station
            robot['target_col'] = simulation_state['observation_station']['col']
            robot['state'] = 'moving_to_observe'
    
    elif robot['state'] == 'moving_to_observe':
        target = simulation_state['observation_station']['col']
        if robot['col'] < target:
            robot['col'] = min(robot['col'] + ROBOT_SPEED, target)
        elif robot['col'] > target:
            robot['col'] = max(robot['col'] - ROBOT_SPEED, target)
        
        if robot['col'] == target:
            robot['state'] = 'observing'
            robot['ticks_remaining'] = OBSERVATION_TICKS
    
    elif robot['state'] == 'observing':
        robot['ticks_remaining'] -= 1
        if robot['ticks_remaining'] <= 0:
            simulation_state['plants_observed'].append({
                'plant_id': robot['holding_plant'],
                'tick': simulation_state['tick'],
                'robot_id': robot['id']
            })
            robot['state'] = 'moving_to_target'
            robot['target_col'] = robot.get('original_col', 0)
    
    elif robot['state'] == 'moving_to_target':
        if robot['target_col'] is not None:
            if robot['col'] < robot['target_col']:
                robot['col'] = min(robot['col'] + ROBOT_SPEED, robot['target_col'])
            elif robot['col'] > robot['target_col']:
                robot['col'] = max(robot['col'] - ROBOT_SPEED, robot['target_col'])
            
            if robot['col'] == robot['target_col']:
                robot['state'] = 'putting_down'
                robot['ticks_remaining'] = PUTDOWN_TICKS
    
    elif robot['state'] == 'putting_down':
        robot['ticks_remaining'] -= 1
        if robot['ticks_remaining'] <= 0:
            if robot['holding_plant']:
                grid = get_sequencer_grid_fn()
                col = int(robot['col'])
                if not grid[robot['row']][col]:
                    grid[robot['row']][col] = robot['holding_plant']
                    set_sequencer_grid_fn(grid)
                robot['holding_plant'] = None
            robot['state'] = 'idle'
            robot['target_col'] = None

In [ ]:
def save_simulation_state(get_h5_file_fn):
    """Persist simulation state to HDF5"""
    import json
    with get_h5_file_fn() as f:
        sim_group = f.require_group("simulation_metadata")
        sim_group.attrs['tick'] = simulation_state['tick']
        sim_group.attrs['running'] = simulation_state['running']
        sim_group.attrs['speed'] = simulation_state['speed']
        sim_group.attrs['robots'] = json.dumps(simulation_state['robots'])
        sim_group.attrs['observation_queue'] = json.dumps(simulation_state['observation_queue'])
        sim_group.attrs['plants_observed'] = json.dumps(simulation_state['plants_observed'])


def load_simulation_state(get_h5_file_fn):
    """Load simulation state from HDF5"""
    import json
    global simulation_state
    try:
        with get_h5_file_fn() as f:
            if "simulation_metadata" in f.keys():
                sim_group = f["simulation_metadata"]
                
                if 'tick' in sim_group.attrs:
                    simulation_state['tick'] = int(sim_group.attrs['tick'])
                if 'running' in sim_group.attrs:
                    simulation_state['running'] = bool(sim_group.attrs['running'])
                if 'speed' in sim_group.attrs:
                    simulation_state['speed'] = float(sim_group.attrs['speed'])
                if 'robots' in sim_group.attrs:
                    simulation_state['robots'] = json.loads(sim_group.attrs['robots'])
                if 'observation_queue' in sim_group.attrs:
                    simulation_state['observation_queue'] = json.loads(sim_group.attrs['observation_queue'])
                if 'plants_observed' in sim_group.attrs:
                    simulation_state['plants_observed'] = json.loads(sim_group.attrs['plants_observed'])
                
                print(f"✓ Loaded simulation state: tick={simulation_state['tick']}, running={simulation_state['running']}")
    except Exception as e:
        print(f"Could not load simulation state: {e}")
        print("Starting with fresh simulation state")